# Kimodo — Nis Dovus Sanati Denemesi

**Amac:** BONES-SEED'de olmayan hareketleri (bokator, muay thai, spinning kick)
Kimodo ile uretmeyi denemek ve gercekten ise yarayip yaramadigini olcmek.

**Neden Colab:** Kimodo metin kodlayici olarak **Llama-3-8B** kullaniyor (~15 GB).
Yerel diskte yer yok + NVIDIA GPU yok. Colab'da ikisi de var.

**Cikti:** MuJoCo qpos CSV → yerelde `scripts/preview_mujoco.py` ile izlenir.

---

### Onkosullar (ikisi de onaylanmis olmali)

| Repo | Durum |
|---|---|
| [meta-llama/Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct) | ACCEPTED |
| [nvidia/Kimodo-G1-RP-v1](https://huggingface.co/nvidia/Kimodo-G1-RP-v1) | gated degil |

> **Runtime → Change runtime type → T4 GPU** sectiginden emin ol.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 1. Kurulum

Kimodo **PyPI'da yok** — GitHub'dan kurulur.

In [ ]:
%pip install -q "git+https://github.com/nv-tlabs/kimodo.git"
print("kurulum tamam")

## 2. HF girisi

Llama-3 gated oldugu icin token sart. **Read-only** token kullan.

In [ ]:
from huggingface_hub import login, hf_hub_download
login()   # token'i buraya yapistir (gizli alan)

# Erisim dogrulamasi — burada patlarsa uretim de patlar
hf_hub_download('meta-llama/Meta-Llama-3-8B-Instruct', 'config.json')
print("Llama-3 erisimi OK")

## 3. Test uretimi

Once modelin calistigini basit bir istemle dogrula.

İlk calistirmada **~16 GB iner** (Llama-3 + Kimodo), birkac dakika surer.
T4'te 16 GB VRAM var ama Llama'yi CPU'ya atmak daha guvenli.

In [ ]:
import os
os.environ["TEXT_ENCODER_DEVICE"] = "cpu"   # VRAM 17GB -> <3GB

!kimodo_gen "a person throws a straight punch" \
    --model Kimodo-G1-RP-v1 \
    --duration 3 --num_samples 1 --seed 0 \
    --output /content/out_test/test.csv

In [ ]:
!find /content/out_test -type f | head

## 4. Cikti formatini olc

Kimodo config'i `G1Skeleton34` diyor — **34 eklem**.
BONES-SEED ve MuJoCo Menagerie `g1.xml` ise **29**.

| Kaynak | Eklem |
|---|---|
| BONES-SEED g1 CSV | 29 |
| Menagerie `g1.xml` | 29 |
| Menagerie `g1_with_hands.xml` | 43 |
| **Kimodo-G1** | **34** |

Fark muhtemelen el/bilek eklemleri. `scripts/import_kimodo.py` bunu
yerelde otomatik kapatir — ama once ne uretildigini gorelim.

In [ ]:
import glob, pandas as pd, numpy as np

for f in sorted(glob.glob('/content/out_test/**/*.csv', recursive=True))[:3]:
    d = pd.read_csv(f)
    print(f'{f}\n  satir={len(d)}  kolon={len(d.columns)}')
    print('  ilk 10 kolon:', list(d.columns[:10]))
    num = d.select_dtypes('number')
    birim = 'radyan' if num.abs().max().max() < 7 else 'derece'
    print(f'  aralik: {num.min().min():.2f} .. {num.max().max():.2f}  -> {birim}\n')

for f in sorted(glob.glob('/content/out_test/**/*.npz', recursive=True))[:2]:
    z = np.load(f, allow_pickle=True)
    print(f, '\n  anahtarlar:', list(z.keys()))
    for k in z.keys():
        try: print(f'    {k}: {z[k].shape}')
        except Exception: pass

## 5. Nis dovus sanati istemleri

**Kritik soru:** Kimodo bokator'u biliyor mu?

Egitim verisi 700 saat studyo mocap — icinde bokator olma ihtimali dusuk.
Bu yuzden her hareketi **iki bicimde** soruyoruz:

1. **Isimle** — model stili biliyorsa en iyisini verir
2. **Tarifle** — bilmiyorsa bile mekanigi tarif ettigimiz icin uretebilir

Ikisi ayni cikarsa model stili bilmiyor demektir.
`control_boxing` ise modelin saglikli calistigini dogrular
(datasette var oldugunu bildigimiz hareket).

In [ ]:
PROMPTS = {
    # --- bokator: isimle vs tarifle ---
    "bokator_named":   "a bokator fighter performs a khmer elbow strike",
    "bokator_desc":    "a fighter steps forward and delivers a downward elbow strike, then recovers to guard",
    "bokator_knee":    "a fighter drives a knee strike upward while pulling both hands down",

    # --- muay thai ---
    "muaythai_named":  "a muay thai fighter throws a roundhouse kick",
    "muaythai_desc":   "a fighter pivots on the left foot and swings the right leg horizontally at head height",

    # --- spinning back kick (formun ornegi) ---
    "spin_named":      "a martial artist performs a spinning back kick",
    "spin_desc":       "a fighter turns 180 degrees on one foot then thrusts the other heel backward, then returns to stance",

    # --- kontrol ---
    "control_boxing":  "a person shadow boxes with alternating straight punches",
}

import subprocess
for name, prompt in PROMPTS.items():
    print(f"\n{'='*65}\n{name}: {prompt}\n{'='*65}")
    subprocess.run([
        "kimodo_gen", prompt,
        "--model", "Kimodo-G1-RP-v1",
        "--duration", "5", "--num_samples", "3", "--seed", "0",
        "--output", f"/content/out/{name}.csv",
    ])

## 6. Ciktilari indir

In [ ]:
!cd /content && zip -qr kimodo_out.zip out && du -sh kimodo_out.zip

from google.colab import files
files.download("/content/kimodo_out.zip")

## 7. Yerelde degerlendirme

```powershell
python scripts/import_kimodo.py kimodo_out.zip
python scripts/preview_mujoco.py --csv data/kimodo_csv/<dosya>.csv --video results/x.mp4
python scripts/contact_sheet.py results/x.mp4
```

Su sorulari yanitla:

| Soru | Anlami |
|---|---|
| `control_boxing` duzgun mu? | Model saglikli calisiyor |
| `*_named` ve `*_desc` farkli mi? | Model stili gercekten biliyor |
| Ikisi de jenerik mi? | Stili bilmiyor → tarif kullan |
| Hareket G1'de dengeli mi? | SONIC egitimine uygun |

**Beklenti:** `bokator_named` muhtemelen jenerik bir dirsek hareketi verecek.
Oyleyse tarif tabanli istemlerle devam etmek daha mantikli.